# 📋 Poblado Inicial del Catálogo de Tickers

**Objetivo:** Llenar la tabla `silver.tickers` con todas las emisoras disponibles en DataBursátil.

**API Endpoint:** `/v2/emisoras`  
**Resultado:** Lista de todas las emisoras del mercado mexicano con su serie

---

## 🔄 Flujo
1. Llamar al endpoint `/v2/emisoras`
2. Parsear respuesta JSON
3. Insertar en `silver.tickers` con `is_active = true`
4. Marcar tickers obsoletos como `is_active = false`

---

**⚠️ Ejecutar solo UNA VEZ** o cuando necesites actualizar el catálogo completo.

## 📦 Setup: Librerías

In [0]:
import requests
import json
from datetime import datetime, timezone

# Para Databricks
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, TimestampType
spark = SparkSession.builder.getOrCreate()

print("✓ Librerías importadas")

## 🔑 Configuración del Token

In [0]:
# ============================================
# CONFIGURACIÓN
# ============================================

# En Databricks, usar secrets:

# Para testing local, comentar línea arriba y descomentar:
TOKEN = "763d9cdce4895c32fcb2f63358c6de"

print(f"🔑 Token: {TOKEN[:10]}...")

## 🌐 Llamar API `/v2/emisoras`

**Documentación:** https://api.databursatil.com/v2/emisoras  
**Respuesta esperada:**
```json
[
  "WALMEX*",
  "AMXL",
  "GFNORTEO",
  "CEMEXCPO",
  ...
]
```

In [0]:
# Construir URL
base_url = "https://api.databursatil.com/v2/emisoras"
params = {"token": TOKEN}

print(f"🌐 Llamando {base_url}...\n")

try:
    response = requests.get(base_url, params=params, timeout=30)
    
    if response.status_code == 200:
        emisoras_raw = response.json()
        
        print(f"✅ Éxito! Recibidas {len(emisoras_raw)} emisoras")
        print(f"   Tamaño respuesta: {len(response.text)} bytes")
        print(f"   Créditos consumidos: ~{len(response.text) / 1024:.2f}\n")
        
        # Debug: Ver tipo de datos
        print(f"🔍 Tipo de respuesta: {type(emisoras_raw)}")
        
        # Si es dict, convertir a lista de keys
        if isinstance(emisoras_raw, dict):
            print(f"   Es un diccionario, extrayendo keys como tickers...")
            emisoras = list(emisoras_raw.keys())
            print(f"   Total emisoras extraídas: {len(emisoras)}\n")
        else:
            emisoras = emisoras_raw
        
        # Mostrar primeras 10 emisoras
        print("📊 Primeras 10 emisoras:")
        for i, emisora in enumerate(emisoras[:10], 1):
            print(f"   {i}. {emisora}")
        
        if len(emisoras) > 10:
            print(f"   ... y {len(emisoras) - 10} más")
        
        api_success = True
        api_error = None
        
    else:
        print(f"❌ Error HTTP {response.status_code}")
        print(f"   Respuesta: {response.text[:200]}")
        emisoras = []
        api_success = False
        api_error = f"HTTP {response.status_code}"
        
except Exception as e:
    print(f"❌ Excepción: {str(e)}")
    import traceback
    traceback.print_exc()
    emisoras = []
    api_success = False
    api_error = str(e)

In [0]:
# DEBUG: Buscar tickers mexicanos conocidos
print(f"\n🔍 Buscando tickers mexicanos en la respuesta:")
tickers_mx = ['WALMEX', 'AMX', 'CEMEX', 'GFNORTE', 'ALFA', 'FEMSA', 'GMEX', 'BIMBO', 'AC', 'TELEVISA']
for ticker_mx in tickers_mx:
    # Buscar coincidencias parciales
    matches = [e for e in emisoras if e and ticker_mx.lower() in str(e).lower()]
    if matches:
        print(f"   {ticker_mx}: {matches[:5]}")  # Mostrar max 5 coincidencias
    else:
        print(f"   {ticker_mx}: ❌ No encontrado")

## 🔄 Transformar a DataFrame

Convertimos la lista de emisoras a un DataFrame con las columnas:
- `ticker`: Símbolo de la emisora
- `is_active`: `true` (todas están activas)
- `first_seen`: Fecha actual
- `last_seen`: Fecha actual

In [0]:
if api_success and emisoras:
    # Preparar datos
    current_time = datetime.now(timezone.utc)
    
    # Filtrar tickers válidos (eliminar null y strings vacíos)
    emisoras_validas = [e for e in emisoras if e and str(e).strip() and str(e) != "null"]
    
    print(f"\n🔍 Filtrado de tickers:")
    print(f"   Total recibidos: {len(emisoras)}")
    print(f"   Tickers válidos: {len(emisoras_validas)}")
    print(f"   Filtrados: {len(emisoras) - len(emisoras_validas)}")
    
    tickers_data = [
        {
            "ticker": str(emisora).strip(),
            "is_active": True,
            "first_seen": current_time,
            "last_seen": current_time
        }
        for emisora in emisoras_validas
    ]
    
    print(f"\n📊 DataFrame preparado:")
    print(f"   Total registros: {len(tickers_data)}")
    print(f"   Columnas: ticker, is_active, first_seen, last_seen")
    print(f"\n   Primeros 10 registros:")
    for i, ticker in enumerate(tickers_data[:10], 1):
        print(f"   {i}. {ticker['ticker']} | active={ticker['is_active']} | first_seen={ticker['first_seen'].date()}")
    
    # Buscar tickers conocidos del mercado mexicano
    print(f"\n🎯 Tickers conocidos encontrados:")
    tickers_conocidos = ['WALMEX', 'WALMEX*', 'AMXL', 'CEMEXCPO', 'GFNORTEO', 'ALFAA', 'AC*']
    for tc in tickers_conocidos:
        encontrado = any(t['ticker'] == tc for t in tickers_data)
        print(f"   {tc}: {'✅ Encontrado' if encontrado else '❌ No encontrado'}")
    
else:
    print("⚠️ No hay datos para transformar")
    tickers_data = []

## 💾 Guardar en `silver.tickers`

**Estrategia MERGE:**
1. Si el ticker ya existe → Actualizar `last_seen` y `is_active = true`
2. Si es nuevo → Insertar con `first_seen = now()`

**Idempotencia:** Puedes ejecutar este notebook múltiples veces sin duplicar registros.

In [0]:
if tickers_data:
    # EN DATABRICKS: Guardar con MERGE
    from delta.tables import DeltaTable
    from pyspark.sql.types import StructType, StructField, StringType, BooleanType, TimestampType
    
    # Crear DataFrame
    schema = StructType([
        StructField("ticker", StringType(), False),
        StructField("is_active", BooleanType(), False),
        StructField("first_seen", TimestampType(), False),
        StructField("last_seen", TimestampType(), False)
    ])
    df = spark.createDataFrame(tickers_data, schema=schema)
    
    # MERGE: Actualizar existing, insertar nuevos
    delta_table = DeltaTable.forName(spark, "silver.tickers")
    
    delta_table.alias("target").merge(
        df.alias("source"),
        "target.ticker = source.ticker"
    ).whenMatchedUpdate(
        set={
            "last_seen": "source.last_seen",
            "is_active": "source.is_active"
        }
    ).whenNotMatchedInsertAll().execute()
    
    print(f"✅ MERGE completado en silver.tickers")
    print(f"   Registros procesados: {len(tickers_data)}")
    
    # Mostrar resumen
    spark.sql("SELECT is_active, COUNT(*) as count FROM silver.tickers GROUP BY is_active").show()
    
    # LOCAL: Guardar como JSON (comentado para Databricks)
    """
    import os
    os.makedirs("output", exist_ok=True)
    with open("output/ticker_catalog.json", "w") as f:
        json.dump(tickers_data, f, indent=2, default=str)
    print(f"✅ LOCAL: Guardado en output/ticker_catalog.json")
    print(f"   Total tickers: {len(tickers_data)}")
    """
    
else:
    print("⚠️ No hay datos para guardar")

## 🔍 Verificación (Solo Databricks)

Ejecuta estas queries para validar el catálogo:

In [0]:
# Total de tickers activos
print("📊 Resumen del catálogo:\n")
spark.sql("""
    SELECT 
        is_active,
        COUNT(*) as total_tickers,
        MIN(first_seen) as oldest_ticker,
        MAX(last_seen) as last_update
    FROM silver.tickers
    GROUP BY is_active
""").show()

# Top 10 tickers
print("\n📋 Top 10 tickers:")
spark.sql("""
    SELECT ticker, is_active, first_seen, last_seen
    FROM silver.tickers
    WHERE is_active = true
    ORDER BY ticker
    LIMIT 10
""").show(truncate=False)

# Tickers específicos para testing
print("\n🎯 Tickers recomendados para testing:")
test_tickers = ["WALMEX", "AMX", "CEMEX", "GFNORTE", "ALFA", "FEMSA", "BIMBO"]
for ticker in test_tickers:
    exists = spark.sql(f"SELECT COUNT(*) as count FROM silver.tickers WHERE ticker = '{ticker}'").collect()[0]['count']
    print(f"   {ticker}: {'✅ Existe' if exists > 0 else '❌ No encontrado'}")

print("\n✅ Notebook completado")

## ✅ Resumen

**Lo que hizo este notebook:**

1. ✅ Llamó a `/v2/emisoras` para obtener todas las emisoras disponibles
2. ✅ Transformó la respuesta JSON a formato DataFrame
3. ✅ Guardó en `silver.tickers` usando MERGE (idempotente)
4. ✅ Marcó todos los tickers como `is_active = true`

**Próximos pasos:**

1. Verifica el catálogo con las queries SQL de arriba
2. Ejecuta `01_ingest_databursatil.ipynb` para ingestar data histórica de TODOS los tickers
3. O ejecuta `01_ingest_single_ticker.ipynb` para probar con un solo ticker

**Frecuencia recomendada:**
- Ejecutar **1 vez al mes** para actualizar el catálogo
- O cuando necesites agregar nuevas emisoras

**Costos:**
- 1 llamada al endpoint `/v2/emisoras`
- ~2-3 créditos de tu cuota de 200,000 mensuales